#RAG Fusion - Get relevant results for **RAG**

In [ ]:
!pip -q install langchain huggingface_hub tiktoken pypdf
!pip -q install google-generativeai chromadb
!pip -q install sentence_transformers

In [ ]:
!pip install langchain -U community

## Download the Data & Utils

In [ ]:
import textwrap
def wrap_text(text, width=90): #preserve_newlines
    # Split the input text into lines based on newline characters
    lines = text.split('\n')

    # Wrap each line individually
    wrapped_lines = [textwrap.fill(line, width=width) for line in lines]

    # Join the wrapped lines back together using newline characters
    wrapped_text = '\n'.join(wrapped_lines)

    return wrapped_text

In [ ]:
import os
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [ ]:
!pip install --upgrade --quiet langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

result = llm.invoke("Write a ballad about LangChain")

print(result.content)

## Imports

In [ ]:
!pip install -q langchain-community langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
import langchain

## Load in Docs

In [ ]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

In [ ]:
from google.colab import userdata
data_path = "/content/Exploring_Places_in_Singapore.pdf"

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(data_path)
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
docs[0]

In [ ]:
print(docs[0].page_content)

In [ ]:
raw_text = ''
for i, doc in enumerate(docs):
    text = doc.page_content
    if text:
        raw_text += text

In [ ]:
print(raw_text)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 100,
    length_function = len,
    is_separator_regex = False,
)

In [ ]:
texts = text_splitter.split_text(raw_text)

In [ ]:
len(texts)

## BGE Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

In [ ]:
model_name = "BAAI/bge-small-en-v1.5"
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

In [ ]:
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    #model_kwargs={'device': 'cuda'},
    encode_kwargs=encode_kwargs,
)

## Vector DB

In [ ]:
### Make the chroma and persiste to disk
db = Chroma.from_texts(texts,embedding_function,persist_directory="./chroma_db")

In [ ]:
query = "Tell me about Universal Studios Singapore?"

db.similarity_search(query, k=5)

## Setup a Retriever

In [ ]:
retriever = db.as_retriever() # can add mmr fetch_k=20, search_type="mmr"

retriever.invoke(query)

## Chat chain

In [ ]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
llm

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
text_reply = chain.invoke("Tell me about Universal Studio Singapore")

print(wrap_text(text_reply))

## With RagFusion

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.prompts import ChatMessagePromptTemplate, PromptTemplate

In [ ]:
prompt = ChatPromptTemplate(input_variables=['original_query'],
                            messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[],template='You are a helpful assistant that generates multiple search queries based on a single input query.')),
                            HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['original_query'], template='Generate multiple search queries related to: {question} \n OUTPUT (4 queries):'))])

In [ ]:
original_query = "universal studios Singapore"

In [ ]:
generate_queries = (
    prompt | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

In [ ]:
generate_queries

In [ ]:
from langchain_core.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results


In [ ]:
ragfusion_chain = generate_queries | retriever.map() | reciprocal_rank_fusion

In [ ]:
langchain.debug = True

In [ ]:
ragfusion_chain.input_schema.schema()

In [ ]:
ragfusion_chain.invoke({"question": original_query})

In [ ]:
from langchain_core.runnables import RunnablePassthrough
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

full_rag_fusion_chain = (
    {
        "context": ragfusion_chain,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
full_rag_fusion_chain.input_schema.schema()

In [ ]:
full_rag_fusion_chain.invoke({"question": "Tell me about Singapore’s nightlife scene?"})